# Foundry IQ over research literature

Every other notebook in this lab retrieves over Contoso's HR handbook and hardware inventory.
This one points the exact same machinery at **20 foundational arXiv papers** on retrieval-augmented
generation, LLM agents, and evaluation — then asks questions that **no single paper answers**.

The corpus is deliberately self-referential: it contains the research that the system you are
watching is built on. RAG (Lewis et al. 2020), GraphRAG, Self-RAG, RAPTOR, ReAct, Toolformer,
AutoGen, Ragas, MT-Bench, and others. You can ask the knowledge base to explain the literature
behind its own architecture.

Nothing here is a mock-up. The PDFs are downloaded live from arXiv, chunked and embedded by
Azure AI Search's managed ingestion, and answered by an LLM-planned multi-query retrieval pipeline
whose full execution trace is printed at the end.

## Step 1: Environment

Keyless throughout. Note that **document embedding is performed by the Search service's own
managed identity**, not by your user account — which is why ingestion works even on subscriptions
where tenant policy blocks direct user access to the Azure OpenAI data plane.

In [1]:
import os
import time

from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_OPENAI_EMBEDDING_MODEL = os.environ.get("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-large")
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]

PAPERS_KS = "arxiv-papers-knowledge-source"
MSLEARN_KS = "mslearn-knowledge-source"
KNOWLEDGE_BASE_NAME = "research-literature-kb"

search_credential = AzureCliCredential(tenant_id=AZURE_TENANT_ID)
print("Environment loaded")

Environment loaded


## Step 2: Download the corpus from arXiv

Twenty papers, fetched from the public arXiv API. No key required. Roughly 30 MB of PDF.

If you re-run this notebook the downloads are skipped for files already on disk. Please keep the
3-second delay — arXiv asks automated clients to rate-limit.

In [2]:
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path

ARXIV_IDS = [
    "2005.11401",  # RAG (Lewis et al.)
    "2404.16130",  # GraphRAG
    "2310.11511",  # Self-RAG
    "2212.10496",  # HyDE
    "2210.03629",  # ReAct
    "2302.04761",  # Toolformer
    "2201.11903",  # Chain-of-Thought
    "2305.14283",  # Query Rewriting for RAG
    "2007.01282",  # Fusion-in-Decoder
    "2004.04906",  # Dense Passage Retrieval
    "2312.10997",  # RAG survey
    "2401.18059",  # RAPTOR
    "2309.15217",  # Ragas
    "2305.15334",  # Gorilla
    "2303.17580",  # HuggingGPT
    "2308.08155",  # AutoGen
    "2402.03367",  # RAG-Fusion
    "2101.00027",  # The Pile
    "2211.09260",  # Task-aware Retrieval with Instructions
    "2306.05685",  # LLM-as-a-Judge / MT-Bench
]

PAPER_DIR = Path("/tmp/arxiv-papers")
PAPER_DIR.mkdir(parents=True, exist_ok=True)

# Look up titles first so the downloaded files are named readably. This matters: the
# filename is what shows up later in the "papers actually cited" output, and
# "2404.16130v2_From_Local_to_Global_A_Graph_RAG_Approach.pdf" is far more useful on a
# slide than "2404.16130.pdf".
metadata_url = "http://export.arxiv.org/api/query?id_list=" + ",".join(ARXIV_IDS) + "&max_results=40"
feed = ET.fromstring(urllib.request.urlopen(metadata_url, timeout=60).read())
namespace = {"a": "http://www.w3.org/2005/Atom"}

papers = []
for entry in feed.findall("a:entry", namespace):
    versioned_id = entry.find("a:id", namespace).text.rsplit("/", 1)[-1]
    title = " ".join(entry.find("a:title", namespace).text.split())
    slug = "".join(c if c.isalnum() or c in " -_" else "" for c in title)[:70].strip().replace(" ", "_")
    papers.append((versioned_id, f"{versioned_id}_{slug}.pdf"))

for versioned_id, filename in papers:
    target = PAPER_DIR / filename
    if target.exists() and target.stat().st_size > 50_000:
        continue
    request = urllib.request.Request(
        f"https://arxiv.org/pdf/{versioned_id}",
        headers={"User-Agent": "foundryiq-research-demo"},
    )
    target.write_bytes(urllib.request.urlopen(request, timeout=90).read())
    print(f"downloaded {filename[:72]}")
    time.sleep(3)  # arXiv asks automated clients to rate-limit

pdfs = sorted(PAPER_DIR.glob("*.pdf"))
print(f"\n{len(pdfs)} papers, {sum(p.stat().st_size for p in pdfs) / 1e6:.0f} MB")

downloaded 2201.11903v6_Chain-of-Thought_Prompting_Elicits_Reasoning_in_Large_Langu


downloaded 2101.00027v1_The_Pile_An_800GB_Dataset_of_Diverse_Text_for_Language_Mode


downloaded 2210.03629v3_ReAct_Synergizing_Reasoning_and_Acting_in_Language_Models.p


downloaded 2404.16130v2_From_Local_to_Global_A_Graph_RAG_Approach_to_Query-Focused_


downloaded 2004.04906v3_Dense_Passage_Retrieval_for_Open-Domain_Question_Answering.


...

downloaded 2303.17580v4_HuggingGPT_Solving_AI_Tasks_with_ChatGPT_and_its_Friends_in


downloaded 2211.09260v2_Task-aware_Retrieval_with_Instructions.pdf


downloaded 2401.18059v1_RAPTOR_Recursive_Abstractive_Processing_for_Tree-Organized_


downloaded 2309.15217v2_Ragas_Automated_Evaluation_of_Retrieval_Augmented_Generatio



20 papers, 31 MB


## Step 3: Create a File knowledge source

`FileKnowledgeSource` handles extraction, chunking, and embedding for you — you upload a PDF and
Azure AI Search does the rest. There is no indexer, skillset, or chunking code to write.

In [3]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    FileKnowledgeSource,
    FileKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceIngestionParameters,
)

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

index_client.create_or_update_knowledge_source(
    FileKnowledgeSource(
        name=PAPERS_KS,
        description=(
            "Foundational arXiv papers on retrieval-augmented generation, "
            "LLM agents, and evaluation"
        ),
        file_parameters=FileKnowledgeSourceParameters(
            ingestion_parameters=KnowledgeSourceIngestionParameters(
                embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                    azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                        resource_url=AZURE_OPENAI_ENDPOINT,
                        deployment_name=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                        model_name=AZURE_OPENAI_EMBEDDING_MODEL,
                    )
                ),
                content_extraction_mode="minimal",
            )
        ),
    )
)
print(f"Knowledge source '{PAPERS_KS}' ready")

Knowledge source 'arxiv-papers-knowledge-source' ready


## Step 4: Ingest the papers

Each upload triggers synchronous chunking and embedding, so this takes a couple of minutes.

**Watch your embedding quota.** A default `text-embedding-3-large` deployment of 30K TPM will
return `429 TooManyRequests` on a corpus this size. Either raise the deployment capacity or rely
on the backoff below.

```bash
az cognitiveservices account deployment create -g <rg> -n <account> \
  --deployment-name text-embedding-3-large --model-name text-embedding-3-large \
  --model-version 1 --model-format OpenAI --sku-name GlobalStandard --sku-capacity 300
```

In [4]:
already_ingested = {f.file_name for f in index_client.list_knowledge_source_files(PAPERS_KS)}
pending = [p for p in pdfs if p.name not in already_ingested]
print(f"{len(already_ingested)} already ingested, {len(pending)} to upload\n")

for pdf in pending:
    for attempt in range(6):
        try:
            uploaded = index_client.upload_knowledge_source_file(
                PAPERS_KS, pdf.read_bytes(), filename=pdf.name
            )
            print(f"  ok  {uploaded.file_size_bytes / 1e6:5.2f} MB  {pdf.name}")
            break
        except Exception as exc:
            if "429" in repr(exc) or "TooManyRequests" in repr(exc):
                delay = 15 * (attempt + 1)
                print(f"  429, backing off {delay}s  {pdf.name}")
                time.sleep(delay)
            else:
                print(f"  FAILED {pdf.name}: {exc!r}"[:160])
                break
    time.sleep(2)

files = list(index_client.list_knowledge_source_files(PAPERS_KS))
print(f"\nCorpus ready: {len(files)} papers indexed")

20 already ingested, 0 to upload




Corpus ready: 20 papers indexed


## Step 5: Create the knowledge base

Two sources: the papers, and the public Microsoft Learn MCP server. The `retrieval_instructions`
keep product documentation from bleeding into research answers.

In [5]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

index_client.create_or_update_knowledge_base(
    KnowledgeBase(
        name=KNOWLEDGE_BASE_NAME,
        description=(
            "Foundational arXiv literature on retrieval-augmented generation, LLM agents, "
            "and evaluation, plus live Microsoft Learn documentation."
        ),
        models=[
            KnowledgeBaseAzureOpenAIModel(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AZURE_OPENAI_ENDPOINT,
                    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
                    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
                )
            )
        ],
        knowledge_sources=[KnowledgeSourceReference(name=n) for n in (PAPERS_KS, MSLEARN_KS)],
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
        output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
        retrieval_instructions=(
            "Use the arXiv papers source for questions about research methods, experimental "
            "results, benchmarks, and claims made in the literature. Always cite which paper a "
            "claim comes from. Use the Microsoft Learn source only for questions about how a "
            "Microsoft product implements something."
        ),
    )
)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' created")

Knowledge base 'research-literature-kb' created


## Step 6: A helper that shows the work

`ask()` prints the synthesized answer, the planner's execution trace including the
**model-generated subqueries**, and which papers the citations actually resolve to.

In [6]:
import collections

from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeSourceParams,
)
from IPython.display import Markdown, display

kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential,
)

FILE_ID_TO_PAPER = {f.file_id: f.file_name for f in index_client.list_knowledge_source_files(PAPERS_KS)}


def ask(question: str):
    request = KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=question)])
        ],
        knowledge_source_params=[
            FileKnowledgeSourceParams(
                knowledge_source_name=PAPERS_KS,
                include_references=True,
                include_reference_source_data=True,
            ),
            KnowledgeSourceParams(
                knowledge_source_name=MSLEARN_KS,
                include_references=True,
                include_reference_source_data=True,
                kind="mcpServer",
            ),
        ],
        include_activity=True,
        max_runtime_in_seconds=180,
    )

    started = time.time()
    result = kb_client.retrieve(retrieval_request=request)
    elapsed = time.time() - started

    display(Markdown(result.response[0].content[0].text))

    print(f"\n--- retrieval trace ({elapsed:.1f}s total) ---")
    for step in result.activity or []:
        detail = step.as_dict()
        subquery = (detail.get("searchIndexArguments") or {}).get("search")
        print(
            f"{detail.get('type', ''):22s} "
            f"{str(detail.get('elapsedMs') or ''):>7s}  "
            f"{detail.get('knowledgeSourceName') or '':30s}"
            f"{'  -> ' + repr(subquery) if subquery else ''}"
        )

    counts = collections.Counter()
    for reference in result.references or []:
        uid = (reference.as_dict().get("sourceData") or {}).get("uid", "")
        paper = FILE_ID_TO_PAPER.get(uid.rsplit("_", 1)[0]) if uid.startswith("file-") else None
        if paper:
            counts[paper] += 1

    if counts:
        print("\n--- papers actually cited ---")
        for paper, chunks in counts.most_common():
            print(f"{chunks:3d} chunks  {paper[:78]}")

    return result

## Question 1: a comparison no single paper makes

GraphRAG and RAPTOR both impose structure over a corpus before retrieval, but they were published
by different groups and neither compares itself to the other. Answering this requires reading both
and contrasting them.

In [7]:
_ = ask(
    "GraphRAG and RAPTOR both impose structure over a corpus before retrieval. "
    "Explain precisely how each builds that structure and how they differ."
)

Both methods precompute **multi-scale structure** over the corpus, but they impose **different kinds of structure** and use them for **different retrieval behaviors**.

## GraphRAG: graph of entities/relations + community hierarchy

GraphRAG first uses an LLM to extract an **entity knowledge graph** from the source documents, where **nodes are entities** and **edges are relationships between entities**.[ref_id:1][ref_id:7] In the extraction pipeline, repeated mentions are aggregated so that entity descriptions are summarized per node, relationships are aggregated into edges, and duplicate relationships contribute to **edge weights**.[ref_id:12]

After the graph is built, GraphRAG applies **hierarchical community detection** to partition the graph into groups of **strongly connected nodes**.[ref_id:12] In the paper’s pipeline, this is done with **Leiden community detection**, applied **recursively** until leaf communities can no longer be partitioned.[ref_id:12] The result is a **hierarchy of mutually exclusive, collectively exhaustive communities** covering the graph at each level.[ref_id:12]

GraphRAG then generates **community summaries** for those communities in a **bottom-up** way, so higher-level summaries recursively incorporate lower-level ones.[ref_id:7][ref_id:15] At query time, GraphRAG does not primarily retrieve original text chunks; instead, it uses **community summaries** to produce **partial answers** in parallel and then combines them with a **reduce-style final summarization** into a global answer.[ref_id:1][ref_id:15][ref_id:19]

So the structure GraphRAG builds is fundamentally **relational and corpus-level**: a **knowledge graph** plus a **hierarchy of graph communities**, with summaries attached to communities.[ref_id:1][ref_id:7][ref_id:12]

## RAPTOR: tree over text chunks built by recursive clustering and summarization

RAPTOR begins by splitting the corpus into **short contiguous text chunks** of about **100 tokens**, while avoiding splitting sentences mid-sentence when possible.[ref_id:13] It then embeds those chunks using **SBERT** embeddings.[ref_id:13]

Using those embeddings, RAPTOR **clusters semantically similar chunks**, then uses an LLM to produce a **summary for each cluster**.[ref_id:0][ref_id:18] Those summaries become parent nodes, and RAPTOR then **repeats the process recursively** — embedding, clustering, and summarizing — to build a **tree from the bottom up** with multiple abstraction levels.[ref_id:0][ref_id:18] In this tree, **leaf nodes** correspond to the original text chunks, while **internal nodes** contain summaries of groups of child nodes.[ref_id:18]

At inference time, RAPTOR retrieves from this **tree**, potentially using nodes from **multiple layers**, so the model can access both **fine-grained evidence** and **higher-level summaries**.[ref_id:0][ref_id:9] The paper explicitly says the full-tree structure matters because retrieval over all layers supports both **detail-oriented** and **higher-order thematic** questions better than restricting retrieval to a single layer.[ref_id:9]

So the structure RAPTOR builds is fundamentally **hierarchical over text segments**: a **summary tree** whose grouping is driven by **semantic similarity of chunk embeddings**.[ref_id:0][ref_id:13][ref_id:18]

## Precise differences

### 1) Basic structural primitive
GraphRAG’s primitive is a **knowledge graph** of **entities and relations**.[ref_id:1][ref_id:7] RAPTOR’s primitive is a **tree** whose leaves are **text chunks** and whose internal nodes are **cluster summaries**.[ref_id:0][ref_id:18]

### 2) What gets grouped
GraphRAG groups **entities** based on **graph connectivity/community structure**.[ref_id:4][ref_id:12] RAPTOR groups **text chunks or summaries** based on **embedding-space semantic similarity**.[ref_id:0][ref_id:13][ref_id:18]

### 3) How hierarchy is formed
GraphRAG’s hierarchy comes from **recursive community detection on a graph**.[ref_id:4][ref_id:12] RAPTOR’s hierarchy comes from **recursive clustering and summarization**, forming a **tree** layer by layer.[ref_id:0][ref_id:18]

### 4) Role of summarization
In GraphRAG, summarization is applied to **graph communities** after graph construction and community partitioning.[ref_id:1][ref_id:12] In RAPTOR, summarization is the mechanism that **creates parent nodes** in the tree at every recursive step.[ref_id:0][ref_id:18]

### 5) Retrieval target at query time
GraphRAG answers queries via **map-reduce over community summaries**, especially to support **global sensemaking** over the corpus.[ref_id:1][ref_id:7][ref_id:15] RAPTOR retrieves **tree nodes across abstraction levels**, including both original chunks and summaries, to answer questions needing either local detail or broader synthesis.[ref_id:0][ref_id:9]

### 6) Intended strength
GraphRAG is motivated by questions that require **global understanding of the whole corpus**, such as main themes or broad trends, where standard vector retrieval is weak.[ref_id:1][ref_id:7] RAPTOR is motivated by the limitation of retrieving only a few short chunks and aims to better support questions that need **integrating information across long documents** at **different levels of abstraction**.[ref_id:0][ref_id:18]

## In one sentence

GraphRAG imposes **explicit relational structure** by extracting a **knowledge graph** and summarizing **graph communities**,[ref_id:1][ref_id:12] whereas RAPTOR imposes **hierarchical semantic structure** by recursively **clustering text embeddings and summarizing clusters into a tree**.[ref_id:0][ref_id:18]

## Practical contrast

If you ask, “What are the major themes across this corpus?”, GraphRAG is designed to answer by aggregating over **community summaries derived from entity-relation structure**.[ref_id:1][ref_id:15] If you ask a question requiring both local details and broad context within long text, RAPTOR is designed to retrieve from **multiple tree layers** containing both the original chunks and their higher-level summaries.[ref_id:9][ref_id:18]

If you want, I can also give you a **side-by-side table** of the two pipelines.


--- retrieval trace (20.1s total) ---
modelQueryPlanning        1720                                
file                       711  arxiv-papers-knowledge-source 
file                       312  arxiv-papers-knowledge-source 
file                       268  arxiv-papers-knowledge-source 
modelAnswerSynthesis     15183                                
agenticReasoning                                              

--- papers actually cited ---
 10 chunks  2404.16130v2_From_Local_to_Global_A_Graph_RAG_Approach_to_Query-Focused_Summar
  6 chunks  2401.18059v1_RAPTOR_Recursive_Abstractive_Processing_for_Tree-Organized_Retrie
  5 chunks  2312.10997v5_Retrieval-Augmented_Generation_for_Large_Language_Models_A_Survey


Expect concrete mechanism, not hand-waving: **Leiden hierarchical community detection** over an
LLM-extracted entity graph with bottom-up community summaries, versus **recursive clustering of
100-token SBERT-embedded chunks** into a summary tree.

## Question 2: a survey across the corpus

Several papers independently attack the question of *when* a model should retrieve rather than
always retrieving. This forces the planner to issue multiple subqueries and gather evidence from
papers that never cite each other.

In [8]:
_ = ask(
    "Several papers propose different ways to decide WHEN a language model should retrieve, "
    "rather than always retrieving. Compare the approaches and say which papers they come from."
)

Several papers propose different mechanisms for deciding **when retrieval should happen only if needed**, instead of retrieving by default.

## 1) SELF-RAG: retrieval decided by a learned special token
**Paper:** *SELF-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection*.[ref_id:1]

SELF-RAG argues that standard RAG often retrieves a fixed number of passages “regardless of whether retrieval is necessary,” which can hurt versatility or lead to unhelpful generations.[ref_id:1] Its solution is to train the model to **adaptively retrieve passages on-demand** by generating special **reflection tokens**.[ref_id:1][ref_id:5]

More specifically, SELF-RAG says the model decides whether retrieval would be helpful given the prompt and prior generation, and if so it emits a **Retrieve** token that triggers retrieval.[ref_id:17][ref_id:8] At inference time, this can be done either as a **hard decision** by predicting `Retrieve`, or by using a **threshold** on the probability of the `Retrieve=Yes` token to trigger retrieval.[ref_id:2][ref_id:7]

So the “when to retrieve” mechanism in SELF-RAG is: **let the LM explicitly predict retrieval need during generation**.[ref_id:2][ref_id:17]

## 2) Active Retrieval-Augmented Generation (ACTIVE-RAG): adaptive retrieval during generation
**Paper:** *Active Retrieval Augmented Generation*.[ref_id:0][ref_id:3]

The retrieved content here does not include the method details from the paper itself, but multiple retrieved documents identify this paper as a prior work specifically about **adaptive / active retrieval** rather than fixed one-shot retrieval.[ref_id:0][ref_id:11] SELF-RAG’s related-work section contrasts prior work that “often retrieves only once at the beginning” with Jiang et al. (2023), which “propose to **adaptively retrieve passages for generation**.”[ref_id:11]

So, based on the retrieved material, ACTIVE-RAG’s contribution in this comparison is that it is a paper on **adaptive retrieval during generation**, though the exact decision rule used to determine *when* retrieval happens was not included in the retrieved excerpts.[ref_id:11]

## 3) FLARE: retrieval triggered by low generation confidence
**Paper:** FLARE.[ref_id:14]

The retrieved survey describes FLARE as a system that **automates retrieval timing by monitoring the confidence of the generation process**, measured using the **probability of generated terms**.[ref_id:14] When that probability drops below a threshold, FLARE **activates retrieval** to gather relevant information.[ref_id:14]

So FLARE’s “when to retrieve” rule is: **retrieve when the model appears uncertain / low-confidence**.[ref_id:14]

---

## Comparison of the approaches

- **SELF-RAG** decides **internally and explicitly** via a learned retrieval token produced by the model itself.[ref_id:2][ref_id:17]
- **FLARE** decides **externally from confidence signals**, triggering retrieval when token probabilities indicate low confidence.[ref_id:14]
- **ACTIVE-RAG** is identified as an **adaptive retrieval during generation** approach, but the retrieved excerpts here do not provide enough detail to state its precise triggering criterion.[ref_id:11]

## Short answer: which papers they come from?

- **Learned retrieval token / reflection-token decision** → *SELF-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection*.[ref_id:1][ref_id:2]
- **Confidence-thresholded retrieval** → *FLARE*.[ref_id:14]
- **Adaptive retrieval during generation** → *Active Retrieval Augmented Generation*.[ref_id:0][ref_id:11]

If you want, I can also turn this into a **table** with columns like **paper / retrieval trigger / training needed / inference behavior**.


--- retrieval trace (12.3s total) ---
modelQueryPlanning        2040                                
file                       323  arxiv-papers-knowledge-source 
file                       269  arxiv-papers-knowledge-source 
file                       246  arxiv-papers-knowledge-source 
modelAnswerSynthesis      8796                                
agenticReasoning                                              

--- papers actually cited ---
  8 chunks  2310.11511v1_Self-RAG_Learning_to_Retrieve_Generate_and_Critique_through_Self-
  6 chunks  2312.10997v5_Retrieval-Augmented_Generation_for_Large_Language_Models_A_Survey
  4 chunks  2305.14283v3_Query_Rewriting_for_Retrieval-Augmented_Large_Language_Models.pdf
  1 chunks  2005.11401v4_Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.


A good answer distinguishes Self-RAG's learned `Retrieve` reflection token (and the ability to
threshold its probability to tune retrieval frequency) from the original RAG architecture, which
simply always retrieves. Watch for it correctly **excluding** Query Rewriting as not being a
"when to retrieve" paper — declining to over-claim is the behaviour you want.

## Question 3: research meets implementation

This one deliberately spans both knowledge sources: the literature on evaluating RAG, and how a
shipping product exposes retrieval telemetry.

In [9]:
_ = ask(
    "According to the papers, what are the main criticisms of using an LLM as a judge to evaluate "
    "generated answers, and what alternative metrics do RAG evaluation papers propose?"
)

According to the papers retrieved, the main criticisms of using an LLM as a judge are about bias, susceptibility to being misled, and evaluation unreliability in some settings.[ref_id:7][ref_id:14][ref_id:17][ref_id:19]

**Main criticisms of LLM-as-a-judge**

- **Position bias:** LLM judges can prefer whichever answer is shown first rather than the actually better answer.[ref_id:14][ref_id:17] In the MT-Bench analysis, all tested LLM judges showed strong position bias, and “most LLM judges favor the first position.”[ref_id:14] For example, GPT-4 was consistent only **65.0%** of the time with the default prompt, while Claude-v1 was consistent only **23.8%** of the time.[ref_id:17]

- **Name bias:** The judged label or assistant name can affect decisions.[ref_id:14][ref_id:17] The paper reports that Claude-v1 showed a name bias favoring “Assistant A.”[ref_id:14][ref_id:17]

- **Verbosity bias:** LLM judges can prefer longer answers even when they are not better.[ref_id:14] The MT-Bench paper defines this as favoring “longer, verbose responses, even if they are not as clear, high-quality, or accurate as shorter alternatives.”[ref_id:14] Under a “repetitive list” attack, failure rates were **91.3%** for Claude-v1 and **91.3%** for GPT-3.5, showing that verbosity can strongly distort judgments.[ref_id:17]

- **Can be misled by flawed context or incorrect candidate answers:** The judge may repeat the same mistake as the answer it is evaluating instead of independently detecting the error.[ref_id:7] One paper states that “LLM makes exactly the same mistake as the given answers in its problem-solving process,” which suggests the judge “may still be misled by the context.”[ref_id:7]

- **Prompt sensitivity / prompt-design dependence:** Judging quality can vary substantially depending on how the evaluation prompt is constructed.[ref_id:7][ref_id:17] In the math-judge example, changing prompting strategy reduced GPT-4’s failure rate from **14/20** with the default prompt to **3/20** with a reference-guided prompt.[ref_id:17]

- **Multi-turn evaluation difficulty:** In multi-turn settings, LLM judges can struggle to track prior turns correctly, causing inaccurate judgments.[ref_id:7] The paper reports that breaking a multi-turn conversation into separate prompts caused the judge to struggle to locate the assistant’s previous response precisely.[ref_id:7]

- **Not always robust enough to be trusted as the sole evaluator:** RAG survey material notes that tools such as RAGAS, ARES, and TruLens use LLMs to score RAG quality, but also emphasizes that RAG evaluation metrics are still not mature or standardized.[ref_id:6][ref_id:0] Another RAG evaluation paper argues that common evaluation setups are inadequate because perplexity is not always predictive of downstream performance, and some common QA benchmarks with short extractive answers may not reflect real use.[ref_id:19]

**Alternative metrics proposed in RAG evaluation papers**

The retrieved papers propose several alternatives or complements to LLM-as-a-judge, often using task metrics, retrieval metrics, or customized RAG-specific metrics.[ref_id:0][ref_id:12][ref_id:15][ref_id:19]

### 1. Traditional answer-quality metrics
These are proposed as direct evaluation metrics for downstream task performance rather than relying only on an LLM judge.[ref_id:0][ref_id:12]

- **Accuracy**[ref_id:0][ref_id:12]
- **Exact Match (EM)**[ref_id:0][ref_id:9][ref_id:15]
- **Recall**[ref_id:0]
- **Precision**[ref_id:0]
- **BLEU**[ref_id:0][ref_id:12][ref_id:15]
- **ROUGE / ROUGE-L**[ref_id:0][ref_id:12][ref_id:15]
- **BERTScore**[ref_id:15]

The survey says QA evaluations often use **EM and F1**, fact-checking often uses **Accuracy**, and BLEU/ROUGE are commonly used for answer quality.[ref_id:9][ref_id:12]

### 2. Retrieval-quality metrics
Several RAG papers propose evaluating retrieval separately instead of scoring only final answers with an LLM judge.[ref_id:0][ref_id:12]

- **Hit Rate**[ref_id:0][ref_id:12]
- **MRR**[ref_id:0][ref_id:12]
- **NDCG**[ref_id:0][ref_id:12]
- **Precision / Recall** for retrieval-related aspects[ref_id:0]

The survey explicitly states that retrieval quality is measured using standard IR metrics such as **Hit Rate, MRR, and NDCG**.[ref_id:12]

### 3. RAG-specific robustness metrics from benchmarks
The RAG survey highlights benchmark-specific metrics tailored to RAG behavior.[ref_id:15]

- **R-Rate (Reappearance Rate):** used in the **RECALL** benchmark for **counterfactual robustness**.[ref_id:15]
- **Benchmark-specific accuracy/EM variants** in **RGB** for **noise robustness, negative rejection, information integration, and counterfactual robustness**.[ref_id:15]

These are presented as alternatives to judging only by an LLM evaluator because they target specific RAG capabilities.[ref_id:15]

### 4. Reference-free or proxy metrics proposed by RAGAS
RAGAS is introduced specifically for settings where **reference answers may not be available**.[ref_id:19] It proposes evaluating different dimensions of a RAG system rather than relying on a single judge score.[ref_id:19][ref_id:15]

The dimensions/metrics cited in the retrieved materials include:

- **Context Relevance**[ref_id:15]
- **Faithfulness**[ref_id:15][ref_id:12]
- **Answer Relevance**[ref_id:15]
- **Cosine Similarity** for one of these dimensions in the survey summary table[ref_id:15]

RAGAS is motivated partly by the limitations of perplexity-based evaluation and the mismatch between short-answer QA evaluation and real RAG use cases.[ref_id:19]

### 5. ARES / TruLens dimension-based metrics
The survey also lists other RAG evaluation tools that score structured dimensions instead of only holistic answer preference.[ref_id:6][ref_id:15]

For **ARES**, the table lists:
- **Context Relevance**
- **Faithfulness**
- **Answer Relevance**
with **Accuracy**-style quantitative metrics.[ref_id:15]

For **TruLens**, the table lists:
- **Context Relevance**
- **Faithfulness**
- **Answer Relevance**
with customized quantitative metrics.[ref_id:15]

### 6. Functional-equivalence metrics in code/API RAG-like evaluation
For generation tasks where many outputs may be valid, one retrieved paper proposes **functional equivalence** rather than string matching or judge preference.[ref_id:11]

- **AST sub-tree matching** is used to detect whether the generated API call matches a valid API pattern in the database.[ref_id:11]
- **Hallucination rate / hallucination identification** is defined based on whether the generated API call matches any known API.[ref_id:11]

This is presented because exact-match style metrics can fail when multiple different API calls are semantically correct.[ref_id:11]

## Bottom line

The papers criticize LLM-as-a-judge mainly because it can be **biased** toward first-position answers or certain names,[ref_id:14][ref_id:17] **favor verbosity over quality**,[ref_id:14][ref_id:17] **be misled by the very answers/context it is supposed to evaluate**,[ref_id:7] and **vary substantially with prompt design and multi-turn presentation**.[ref_id:7][ref_id:17]

As alternatives, RAG evaluation papers propose using **task metrics** like **Accuracy, EM, BLEU, ROUGE, and BERTScore**,[ref_id:0][ref_id:12][ref_id:15] **retrieval metrics** like **Hit Rate, MRR, and NDCG**,[ref_id:0][ref_id:12] **RAG-specific robustness metrics** such as **R-Rate** and benchmark-specific robustness scores,[ref_id:15] and **dimension-based metrics** such as **Context Relevance, Faithfulness, and Answer Relevance** in frameworks like RAGAS, ARES, and TruLens.[ref_id:15][ref_id:6][ref_id:19]

If you want, I can also reorganize this into a **two-column table: “Criticism” vs. “Alternative metric/paper.”**


--- retrieval trace (21.5s total) ---
modelQueryPlanning        1434                                
file                       289  arxiv-papers-knowledge-source 
file                       259  arxiv-papers-knowledge-source 
file                       253  arxiv-papers-knowledge-source 
modelAnswerSynthesis     18531                                
agenticReasoning                                              

--- papers actually cited ---
  8 chunks  2312.10997v5_Retrieval-Augmented_Generation_for_Large_Language_Models_A_Survey
  3 chunks  2306.05685v4_Judging_LLM-as-a-Judge_with_MT-Bench_and_Chatbot_Arena.pdf
  2 chunks  2310.11511v1_Self-RAG_Learning_to_Retrieve_Generate_and_Critique_through_Self-
  2 chunks  2404.16130v2_From_Local_to_Global_A_Graph_RAG_Approach_to_Query-Focused_Summar
  2 chunks  2309.15217v2_Ragas_Automated_Evaluation_of_Retrieval_Augmented_Generation.pdf
  2 chunks  2305.15334v1_Gorilla_Large_Language_Model_Connected_with_Massive_APIs.pdf
  1 chunks  2305.14

## An honest caveat

Ask *"which papers report results on multi-hop QA and what datasets did they use?"* and the system
will confidently name **HotpotQA**, **MuSiQue**, and **2WikiMultiHopQA** papers — none of which are
in this corpus. It picked them up from the **bibliographies and related-work sections** of papers
that are.

This is a real and underappreciated failure mode for RAG over academic PDFs: a reference list is
lexically dense with paper titles, so it retrieves well while carrying almost no actual content.
Chunk-level provenance is intact and the citations are truthful — the chunks really do contain
those strings — but the *inference* that these are corpus papers is wrong.

Mitigations worth discussing: strip reference sections at ingestion, filter chunks by section
heading, or constrain the answer to a known document allow-list. It is a good, concrete illustration
of why retrieval provenance and corpus hygiene matter more than model quality in production RAG.

## What to take away

- The same knowledge base machinery used for enterprise documents works unmodified on research
  literature — 20 PDFs, no chunking code, no vector database to operate.
- The planner **decomposes** a comparative question into targeted subqueries and gathers evidence
  across papers that do not cite each other. The trace shows exactly which queries it invented.
- Citations resolve to specific papers and chunk counts, so every claim is auditable.
- The bibliography failure mode above is a reminder that corpus hygiene, not model choice, is
  usually the binding constraint on RAG quality.